In [ ]:
# CELL 1
# ERA5 UNIFIED DOWNLOAD — SINGLE PASS, ALL 18 VARIABLES
# ============================================================================
# Reference implementation for reproducing the ERA5 dataset from scratch.
# Combines the buffer-domain fix, the full flood-relevant variable set
# (including soil layers 2/3 and soil_type), and transparent handling of
# CDS's ZIP-splitting behaviour into ONE script — no separate soil download
# or merge step needed, since this downloads everything together from the
# start rather than adding variables after an earlier download completed.
# ============================================================================
import os
import calendar
import zipfile
import tempfile
import cdsapi
import xarray as xr
from tqdm import tqdm

OUT_DIR = "."

# ── DOMAIN: 50-MILE BUFFER RETAINED ──
# 50 miles ≈ 80.5 km. At UK latitudes (~55°N), longitude degrees are
# physically shorter than latitude degrees (111 km/° lat vs ~64 km/° lon,
# since east-west distance shrinks with cos(latitude)). PAD=1.5° gives
# ~104 miles buffer in latitude and ~60 miles in longitude — safely over
# 50 miles in both directions.
PAD = 1.5
UK_AREA_ORIGINAL = [60.9, -8.2, 49.9, 1.8]  # [N, W, S, E]
UK_AREA = [60.9 + PAD, -8.2 - PAD, 49.9 - PAD, 1.8 + PAD]
UK_AREA = [round(v * 4) / 4 for v in UK_AREA]  # round to nearest 0.25° grid step

# ── VARIABLES: all 18 in one request ──
VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",
    "2m_temperature",
    "mean_sea_level_pressure",
    "sea_surface_temperature",
    "surface_pressure",
    "skin_temperature",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "volumetric_soil_water_layer_3",
    "geopotential",
    "convective_available_potential_energy",  # CAPE — convective instability
    "total_column_water_vapour",              # TCWV — moisture available to fall as rain
    "land_sea_mask",                          # authoritative land/sea flag
    "convective_precipitation",               # cross-check vs IMERG
    "boundary_layer_height",                  # convective development
    "soil_type",                              # static FAO soil classification (infiltration capacity)
]

# Config tag baked into every filename: encodes PAD and variable count, so
# any future change to either automatically produces new filenames rather
# than silently reusing stale cached files from a different configuration
# (the exact bug that caused a wasted 30-month re-download earlier).
CONFIG_TAG = f"pad{PAD}_v{len(VARIABLES)}"

TIME_RANGE = [(y, m) for y in (2024, 2025) for m in range(1, 13)] + [(2026, m) for m in range(1, 7)]
# 2026 stops at June — a few weeks behind the download date, avoiding
# requests for days not yet in the archive. Months from roughly May/June
# 2026 onward are served as ERA5T (preliminary) rather than final ERA5;
# this is standard practice, just worth noting for the methodology writeup.


def download_month(client, year, month):
    month_str = f"{month:02d}"
    out_path = os.path.join(OUT_DIR, f"era5_uk_{CONFIG_TAG}_{year}_{month_str}.nc")
    if os.path.exists(out_path):
        return "skipped"

    days_in_month = calendar.monthrange(year, month)[1]
    request = {
        "product_type": "reanalysis",
        "variable": VARIABLES,
        "year": str(year),
        "month": month_str,
        "day": [f"{d:02d}" for d in range(1, days_in_month + 1)],
        "time": [f"{h:02d}:00" for h in range(24)],
        "area": UK_AREA,
        "format": "netcdf",
    }
    try:
        client.retrieve("reanalysis-era5-single-levels", request).download(out_path)
        return "ok"
    except Exception as e:
        if os.path.exists(out_path):
            os.remove(out_path)
        return f"failed: {str(e)[:80]}"


def open_month(path):
    """Transparently handle CDS's ZIP-splitting: a request mixing variables
    with different time semantics (instantaneous vs accumulated, e.g. wind
    speed vs convective_precipitation) gets returned as a ZIP containing two
    netCDF files instead of one plain netCDF, even though 'format': 'netcdf'
    was requested. This detects and merges the split transparently, so it
    never needs a separate "fix" step after the fact."""
    with open(path, "rb") as f:
        header = f.read(4)

    if header[:2] == b"PK":  # ZIP signature
        with tempfile.TemporaryDirectory() as tmpdir:
            with zipfile.ZipFile(path) as z:
                z.extractall(tmpdir)
                names = z.namelist()
            parts = [xr.open_dataset(os.path.join(tmpdir, n)) for n in names]
            ds = xr.merge(parts, compat="override", join="outer")
            ds.load()  # materialize before the temp files are deleted
            for p in parts:
                p.close()
    else:
        ds = xr.open_dataset(path)

    if "valid_time" in ds.dims and "time" in ds.dims:
        ds = ds.drop_dims("time").rename({"valid_time": "time"})
    elif "valid_time" in ds.dims:
        ds = ds.rename({"valid_time": "time"})

    return ds


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    print("=" * 70)
    print("ERA5 UNIFIED DOWNLOAD — 18 variables, single pass")
    print("=" * 70)
    print(f"Buffered domain: {UK_AREA}  (original: {UK_AREA_ORIGINAL}, +{PAD}° buffer)")
    print(f"Variables: {len(VARIABLES)}")
    print(f"Months: {len(TIME_RANGE)} ({TIME_RANGE[0][0]}-{TIME_RANGE[0][1]:02d} to "
          f"{TIME_RANGE[-1][0]}-{TIME_RANGE[-1][1]:02d})\n")

    client = cdsapi.Client()
    results = {"ok": 0, "skipped": 0, "failed": 0}

    with tqdm(TIME_RANGE, desc="Downloading", unit="month") as pbar:
        for year, month in pbar:
            pbar.set_postfix_str(f"{year}-{month:02d}")
            status = download_month(client, year, month)
            key = "failed" if status.startswith("failed") else status
            results[key] = results.get(key, 0) + 1
            if status.startswith("failed"):
                tqdm.write(f"  ✗ {year}-{month:02d}: {status}")

    print(f"\nDownload summary: {results}")

    # ── Combine all months, handling ZIP-splitting transparently per month ──
    final_path = os.path.join(OUT_DIR, f"era5_uk_2024_2026_{CONFIG_TAG}_FINAL.nc")
    if os.path.exists(final_path):
        print(f"✓ Final file already exists: {final_path}")
        return

    print("\nCombining months (auto-handling any ZIP-split files)...")
    monthly_datasets = []
    for year, month in TIME_RANGE:
        path = os.path.join(OUT_DIR, f"era5_uk_{CONFIG_TAG}_{year}_{month:02d}.nc")
        if os.path.exists(path):
            monthly_datasets.append(open_month(path))

    print(f"  {len(monthly_datasets)} months loaded, concatenating...")
    combined = xr.concat(monthly_datasets, dim="time").sortby("time")

    print(f"  Total timesteps: {len(combined.time)}")
    print(f"  Grid: {len(combined.latitude)} lat x {len(combined.longitude)} lon")
    print(f"  Variables ({len(combined.data_vars)}): {list(combined.data_vars)}")

    encoding = {var: {"zlib": True, "complevel": 4} for var in combined.data_vars}
    combined.to_netcdf(final_path, encoding=encoding)
    for d in monthly_datasets:
        d.close()

    size_gb = os.path.getsize(final_path) / 1e9
    print(f"\n✓ Saved: {final_path} ({size_gb:.2f} GB)")


if __name__ == "__main__":
    main()

In [15]:
import xarray as xr

ds = xr.open_dataset("era5_uk_2024_2026_hourly_pad1.5.nc")
print(f"Latitude range: {ds.latitude.values.min()} to {ds.latitude.values.max()}")
print(f"Longitude range: {ds.longitude.values.min()} to {ds.longitude.values.max()}")
ds.close()

Latitude range: 48.5 to 62.5
Longitude range: -9.75 to 3.25


In [ ]:
# CELL2
# DIAGNOSE: what format are the downloaded .nc files actually in?
# Does NOT delete or modify anything — read-only inspection.
import os
import glob
import zipfile

files = sorted(glob.glob("era5_uk_*_hourly.nc"))
print(f"Found {len(files)} files matching pattern")

if not files:
    print("No files found — check you're running this from the right directory (THESIS_PROJECT).")
else:
    for path in files[:3]:  # just check the first 3 — if one is a ZIP, they all likely are
        size_mb = os.path.getsize(path) / 1e6
        with open(path, "rb") as f:
            header = f.read(8)

        print(f"\n{path} ({size_mb:.1f} MB)")
        print(f"  First 8 bytes (hex): {header.hex()}")

        if header[:2] == b"PK":
            print("  -> This is a ZIP file (starts with 'PK'), not a raw netCDF/GRIB file.")
            with zipfile.ZipFile(path) as z:
                print(f"  -> Contents: {z.namelist()}")
        elif header[:3] == b"CDF":
            print("  -> This IS a valid netCDF3 file.")
        elif header[:4] == b"\x89HDF":
            print("  -> This IS a valid netCDF4/HDF5 file.")
        elif header[:4] == b"GRIB":
            print("  -> This is actually a GRIB file, not netCDF, despite the .nc extension/request.")
        else:
            print(f"  -> Unrecognized format. Raw header: {header}")

Found 32 files matching pattern

era5_uk_pad1.5_2024_01_hourly.nc (42.8 MB)
  First 8 bytes (hex): 504b030414000000
  -> This is a ZIP file (starts with 'PK'), not a raw netCDF/GRIB file.
  -> Contents: ['data_stream-oper_stepType-instant.nc', 'data_stream-oper_stepType-accum.nc']

era5_uk_pad1.5_2024_02_hourly.nc (40.2 MB)
  First 8 bytes (hex): 504b030414000000
  -> This is a ZIP file (starts with 'PK'), not a raw netCDF/GRIB file.
  -> Contents: ['data_stream-oper_stepType-instant.nc', 'data_stream-oper_stepType-accum.nc']

era5_uk_pad1.5_2024_03_hourly.nc (43.0 MB)
  First 8 bytes (hex): 504b030414000000
  -> This is a ZIP file (starts with 'PK'), not a raw netCDF/GRIB file.
  -> Contents: ['data_stream-oper_stepType-instant.nc', 'data_stream-oper_stepType-accum.nc']


In [18]:
# CELL 3 reading dataset
import h5py
import os
import numpy as np

hdf5_files = [f for f in os.listdir('GPM_DATA') if f.endswith('.nc4')]

if hdf5_files:
    filename = os.path.join('GPM_DATA', hdf5_files[0])   # ← full path fix

    with h5py.File(filename, 'r') as f:
        datasets = []
        for name, obj in f.items():
            if isinstance(obj, h5py.Dataset):
                datasets.append((name, obj.shape, obj.size))

        if datasets:
            print("\n📊 Dataset Summary:")
            print("-" * 70)
            for name, shape, size in sorted(datasets, key=lambda x: -x[2])[:5]:
                print(f"Name: {name}")
                print(f"  Shape: {shape}")
                print(f"  Size: {size} elements")

                ds = f[name]
                if ds.size > 0 and ds.dtype.kind in ['f', 'i', 'u']:
                    data_sample = ds[()]
                    print(f"  Data type: {ds.dtype}")
                    print(f"  Min: {np.nanmin(data_sample):.6f}")
                    print(f"  Max: {np.nanmax(data_sample):.6f}")
                    print(f"  Mean: {np.nanmean(data_sample):.6f}")
                    print(f"  Sample values (first 5): {data_sample.flat[:5]}")
                print()
else:
    print("No .nc4 files found in GPM_DATA/")


📊 Dataset Summary:
----------------------------------------------------------------------
Name: MWprecipitation
  Shape: (1, 100, 110)
  Size: 11000 elements
  Data type: float32
  Min: -9999.900391
  Max: 6.890000
  Mean: -7749.877930
  Sample values (first 5): [-9999.9 -9999.9 -9999.9 -9999.9 -9999.9]

Name: MWprecipSource
  Shape: (1, 100, 110)
  Size: 11000 elements
  Data type: int16
  Min: 0.000000
  Max: 3.000000
  Mean: 0.675000
  Sample values (first 5): [0 0 0 0 0]

Name: MWobservationTime
  Shape: (1, 100, 110)
  Size: 11000 elements
  Data type: int16
  Min: -9999.000000
  Max: 14.000000
  Mean: -7746.560545
  Sample values (first 5): [-9999 -9999 -9999 -9999 -9999]

Name: IRprecipitation
  Shape: (1, 100, 110)
  Size: 11000 elements
  Data type: float32
  Min: -9999.900391
  Max: 5.080000
  Mean: -1181.739136
  Sample values (first 5): [0.02       0.17999999 0.22999999 0.39       0.9       ]

Name: IRinfluence
  Shape: (1, 100, 110)
  Size: 11000 elements
  Data type: int

# Understanding GPM 3IMERGHHE Data

## What is GPM 3IMERGHHE?

**GPM** = Global Precipitation Measurement  
**3IMERGHHE** = Integrated Multi-satellite Retrievals for GPM (IMERG) Half-Hourly Early  
**V07** = Version 7 (the current version)


In [19]:
# CELL 4: Explore all datasets and their meanings
import numpy as np
import os
import h5py

print("="*80)
print("GPM 3IMERGHHE DATA DICTIONARY - All Variables in Your File")
print("="*80)

hdf5_files = [f for f in os.listdir('GPM_DATA') if f.endswith('.nc4')]

if hdf5_files:
    filename = os.path.join('GPM_DATA', hdf5_files[0])   # ← full path fix

    with h5py.File(filename, 'r') as f:
        datasets_info = []

        def collect_info(name, obj):
            if isinstance(obj, h5py.Dataset):
                datasets_info.append({
                    'path': name,
                    'shape': obj.shape,
                    'dtype': obj.dtype,
                    'size': obj.size,
                    'dataset_obj': obj
                })

        f.visititems(collect_info)
        datasets_info.sort(key=lambda x: x['path'])

        print(f"\nFile: {filename}")
        print(f"Total Datasets: {len(datasets_info)}\n")

        # Data dictionary — matches YOUR actual downloaded variables (Early-Run product)
        var_descriptions = {
            'precipitation': 'Combined multi-satellite precipitation estimate (mm/hr)',
            'precipitation_cnt': 'Count of valid precipitation observations',
            'precipitation_cnt_cond': 'Conditional count of precipitation observations',
            'MWprecipitation': 'Microwave-only precipitation estimate (mm/hr)',
            'MWprecipitation_cnt': 'Count of microwave precipitation observations',
            'MWprecipitation_cnt_cond': 'Conditional count of MW precipitation observations',
            'MWprecipSource': 'Source sensor for microwave precipitation estimate (categorical code)',
            'MWobservationTime': 'Time of microwave observation within the half-hour window (minutes)',
            'IRprecipitation': 'Infrared-only precipitation estimate (mm/hr)',
            'IRinfluence': 'Weight/influence of infrared estimate in final blend (0-100%)',
            'precipitationQualityIndex': 'Precipitation quality index (0-100%)',
            'probabilityLiquidPrecipitation': 'Probability of liquid (vs frozen) precipitation (0-100%)',
            'randomError': 'Random error estimate (mm/hr)',
            'time': 'Timestamp for this granule',
            'time_bnds': 'Time bounds (start/end) for this granule',
            'lat': 'Latitude coordinates',
            'lon': 'Longitude coordinates',
        }

        for i, info in enumerate(datasets_info, 1):
            path = info['path']
            shape = info['shape']
            dtype = info['dtype']
            var_name = path.split('/')[-1]
            description = var_descriptions.get(var_name, 'Unknown variable')

            print(f"{i}. {path}")
            print(f"   Shape: {shape}")
            print(f"   Type: {dtype}")
            print(f"   Size: {info['size']:,} elements")
            print(f"   Description: {description}")

            if info['dataset_obj'].dtype.kind in ['f', 'i', 'u']:
                data_sample = info['dataset_obj'][()]
                # NOTE: raw fill values (-9999/-9999.9) are NOT yet masked here —
                # this is a raw diagnostic view, so ranges below may include them.
                # np.isnan alone won't catch them since they're not actually NaN in the raw file.
                print(f"   Raw min/max (fill values NOT masked): "
                      f"{np.min(data_sample):.2f} to {np.max(data_sample):.2f}")
            print()
else:
    print("No .nc4 files found in GPM_DATA/")

print("\n" + "="*80)
print("INTERPRETATION GUIDE")
print("="*80)
print("""
📍 COORDINATES:
  - lat, lon: Geographic coordinates for each grid cell
  - Each grid cell: 0.1° × 0.1° resolution (~11 km × 11 km)

🌧️ PRECIPITATION ESTIMATES:
  - precipitation: Primary combined rainfall estimate (use this for most analysis)
  - MWprecipitation / IRprecipitation: Individual sensor-type estimates
  - Range: 0-100+ mm/hr (0 = no rain)

📊 QUALITY INDICATORS:
  - IRinfluence: How much infrared satellites contributed to the blend (0-100%)
  - precipitationQualityIndex: Overall data quality (0-100%)
  - Higher values = more reliable estimates

⚠️ SPECIAL VALUES (IMPORTANT):
  - -9999 / -9999.9: Fill value, meaning NO DATA (not a real observation)
  - These are NOT automatically converted to NaN when read via h5py/xarray —
    they must be explicitly masked (see read_imerg() fix) before any analysis.
  - 0: Genuine "no precipitation" (a real, valid observation of zero rain)

⏰ TEMPORAL COVERAGE:
  - Each file: 30-minute snapshot
  - Combine multiple files for time series analysis
  - File naming includes start/end time (UTC)
""")

GPM 3IMERGHHE DATA DICTIONARY - All Variables in Your File

File: GPM_DATA/3B-HHR-E.MS.MRG.3IMERG.20240820-S033000-E035959.0210.V07B.HDF5.SUB.nc4
Total Datasets: 13

1. IRinfluence
   Shape: (1, 100, 110)
   Type: int16
   Size: 11,000 elements
   Description: Weight/influence of infrared estimate in final blend (0-100%)
   Raw min/max (fill values NOT masked): -9999.00 to 34.00

2. IRprecipitation
   Shape: (1, 100, 110)
   Type: float32
   Size: 11,000 elements
   Description: Infrared-only precipitation estimate (mm/hr)
   Raw min/max (fill values NOT masked): -9999.90 to 5.08

3. MWobservationTime
   Shape: (1, 100, 110)
   Type: int16
   Size: 11,000 elements
   Description: Time of microwave observation within the half-hour window (minutes)
   Raw min/max (fill values NOT masked): -9999.00 to 14.00

4. MWprecipSource
   Shape: (1, 100, 110)
   Type: int16
   Size: 11,000 elements
   Description: Source sensor for microwave precipitation estimate (categorical code)
   Raw min/max 

In [4]:
# CELL 5 (v3): LOAD BUFFERED ERA5 AND UPSAMPLE TO 0.1° — MEMORY-SAFE BATCHED VERSION
# ============================================================================
# FIXES A MEMORY CRASH: the previous version accumulated every upsampled
# timestep in a Python list across all 21,888 timesteps before writing
# anything to disk (~25+ GB in RAM), crashing partway through. This version
# processes BATCH_SIZE timesteps at a time, writes each batch to its own
# small file immediately, then combines all batches at the end — never
# holding more than one batch's worth of data in memory. Same pattern
# already used successfully for the ERA5 monthly downloads.
# ============================================================================
import os
import glob
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm

ERA5_PATH = "era5_uk_2024_2026_hourly_pad1.5_full.nc"
BATCH_DIR = "era5_upsample_batches"
os.makedirs(BATCH_DIR, exist_ok=True)

UK_BOUNDS_FINAL = {"lat_min": 48.5, "lat_max": 62.5, "lon_min": -9.75, "lon_max": 3.25}
UPSAMPLED_ERA5 = "era5_uk_upsampled_01deg_buffered.nc"

BATCH_SIZE = 500  # ~500 timesteps x ~16 variables x target grid size stays well within RAM

print(f"Loading buffered ERA5 from {ERA5_PATH} ...")
ds_era5_full = xr.open_dataset(ERA5_PATH)
time_dim = 'valid_time' if 'valid_time' in ds_era5_full.dims else 'time'
lat_desc = ds_era5_full["latitude"].values[0] > ds_era5_full["latitude"].values[-1]

ds_era5 = ds_era5_full.sel(
    latitude=slice(UK_BOUNDS_FINAL["lat_max"], UK_BOUNDS_FINAL["lat_min"]) if lat_desc
    else slice(UK_BOUNDS_FINAL["lat_min"], UK_BOUNDS_FINAL["lat_max"]),
    longitude=slice(UK_BOUNDS_FINAL["lon_min"], UK_BOUNDS_FINAL["lon_max"]),
)

original_time = ds_era5[time_dim].values
n_timesteps = len(original_time)

new_lat = np.arange(UK_BOUNDS_FINAL["lat_min"], UK_BOUNDS_FINAL["lat_max"] + 0.05, 0.1)
new_lon = np.arange(UK_BOUNDS_FINAL["lon_min"], UK_BOUNDS_FINAL["lon_max"] + 0.05, 0.1)

era5_vars_list = [v for v in ds_era5.data_vars if v not in ['number', 'expver']]
static_vars = [v for v in era5_vars_list if time_dim not in ds_era5[v].dims]
tvar_vars = [v for v in era5_vars_list if time_dim in ds_era5[v].dims]

# land_sea_mask is physically constant (land/sea boundaries don't change),
# even though it may technically carry a time dimension in the downloaded
# file (an artifact of requesting it per-timestep alongside time-varying
# fields). Treating it as time-varying would re-interpolate the exact same
# values 21,888 times for no benefit — force it into static_vars instead.
if "lsm" in tvar_vars:
    tvar_vars.remove("lsm")
    static_vars.append("lsm")

print(f"Static variables: {static_vars}")
print(f"Time-varying variables: {tvar_vars}")
print(f"Total timesteps: {n_timesteps:,}, batch size: {BATCH_SIZE}\n")


def interp_var(data_array):
    # Linear interpolation — meaningfully faster than cubic, and for a
    # 0.25° -> 0.1° upsampling (not a large resolution jump), the accuracy
    # difference vs cubic is small relative to the time savings.
    return data_array.interp(latitude=new_lat, longitude=new_lon, method="linear",
                              kwargs={"bounds_error": False})


# ── Static variables: upsample once, save separately ──
static_path = os.path.join(BATCH_DIR, "static_vars.nc")
if not os.path.exists(static_path):
    print("Upsampling static variables...")
    ds_static = xr.Dataset(coords={"latitude": new_lat, "longitude": new_lon})
    for var_name in static_vars:
        source_array = ds_era5[var_name]
        if time_dim in source_array.dims:
            # e.g. lsm — technically has a time dim in the file, but the
            # values are constant, so just take the first timestep
            source_array = source_array.isel({time_dim: 0})
        ds_static[var_name] = interp_var(source_array).astype("float32")
    ds_static.to_netcdf(static_path)
    ds_static.close()
    print(f"✓ Saved static variables -> {static_path}")

# ── Time-varying variables: batched, written incrementally ──
n_batches = int(np.ceil(n_timesteps / BATCH_SIZE))
print(f"\nProcessing {n_timesteps:,} timesteps in {n_batches} batches of {BATCH_SIZE}...")

for batch_idx in tqdm(range(n_batches), desc="Batches", unit="batch"):
    batch_path = os.path.join(BATCH_DIR, f"batch_{batch_idx:04d}.nc")
    if os.path.exists(batch_path):
        continue  # already done — safe to re-run after a crash, no lost progress this time

    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, n_timesteps)
    batch_time = original_time[start:end]

    ds_batch = xr.Dataset(coords={"time": batch_time, "latitude": new_lat, "longitude": new_lon})
    for var_name in tvar_vars:
        # Select the WHOLE batch's time range at once, then interpolate in
        # ONE call — xarray broadcasts the lat/lon interpolation over the
        # time dimension automatically. This is the key fix: the previous
        # version called interp() once PER TIMESTEP (500 calls per
        # variable per batch), paying ~0.68s of per-call overhead each
        # time — 90+ minutes per batch. This does it in ONE call per
        # variable per batch instead, cutting the number of interp() calls
        # by ~500x.
        batch_slice = ds_era5[var_name].isel({time_dim: slice(start, end)})
        ds_batch[var_name] = interp_var(batch_slice).astype("float32")

    encoding = {var: {"zlib": True, "complevel": 4} for var in ds_batch.data_vars}
    ds_batch.to_netcdf(batch_path, encoding=encoding)
    ds_batch.close()

print(f"\n✓ All batches written to {BATCH_DIR}/")

# ── Combine all batches + static variables into the final file ──
if os.path.exists(UPSAMPLED_ERA5):
    print(f"✓ Final file already exists: {UPSAMPLED_ERA5}")
else:
    print("\nCombining batches into final file...")
    batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, "batch_*.nc")))
    batch_datasets = [xr.open_dataset(f) for f in batch_files]
    ds_combined = xr.concat(batch_datasets, dim="time").sortby("time")

    ds_static = xr.open_dataset(static_path)
    ds_combined = xr.merge([ds_combined, ds_static], compat="override", join="inner")

    print(f"  Total: {len(ds_combined.time)} timesteps")
    print(f"  Variables: {list(ds_combined.data_vars)}")

    encoding = {var: {"zlib": True, "complevel": 4} for var in ds_combined.data_vars}
    ds_combined.to_netcdf(UPSAMPLED_ERA5, encoding=encoding)
    for d in batch_datasets:
        d.close()
    ds_static.close()

    size_gb = os.path.getsize(UPSAMPLED_ERA5) / 1e9
    print(f"\n✓ Saved: {UPSAMPLED_ERA5} ({size_gb:.2f} GB)")

# ── Load final result for downstream use ──
ds_era5 = xr.open_dataset(UPSAMPLED_ERA5)
era5_lat = ds_era5["latitude"].values
era5_lon = ds_era5["longitude"].values
era5_time = pd.to_datetime(ds_era5["time"].values)
era5_vars = list(ds_era5.data_vars)
n_era5_cells = len(era5_lat) * len(era5_lon)

print(f"\n✓ ERA5 grid: {len(era5_lat)} lat x {len(era5_lon)} lon = {n_era5_cells:,} cells")
print(f"✓ ERA5 time steps: {len(era5_time):,}")
print(f"✓ ERA5 variables: {era5_vars}")

Loading buffered ERA5 from era5_uk_2024_2026_hourly_pad1.5_full.nc ...
Static variables: ['slt', 'lsm']
Time-varying variables: ['u10', 'v10', 'd2m', 't2m', 'msl', 'sst', 'sp', 'skt', 'swvl1', 'z', 'cape', 'tcwv', 'blh', 'cp', 'swvl2', 'swvl3']
Total timesteps: 21,888, batch size: 500


Processing 21,888 timesteps in 44 batches of 500...


Batches: 100%|██████████| 44/44 [10:03<00:00, 13.72s/batch]



✓ All batches written to era5_upsample_batches/

Combining batches into final file...


: 

In [ ]:
# CELL 6: MEMORY-SAFE COMBINE OF UPSAMPLED BATCHES INTO FINAL FILE
# ============================================================================
# Replaces the xr.concat-based combine step, which tried to hold all 44
# batches (~26GB) in memory simultaneously — the same category of crash as
# the original per-timestep interpolation loop, just relocated to a later
# step. This uses the netCDF4 library directly to append each batch's data
# onto the output file's unlimited time dimension, one batch at a time,
# never holding more than one batch (~590MB) in memory at once.
# ============================================================================
import os
import glob
import numpy as np
import netCDF4 as nc4
import xarray as xr

BATCH_DIR = "era5_upsample_batches"
STATIC_PATH = os.path.join(BATCH_DIR, "static_vars.nc")
FINAL_PATH = "era5_uk_upsampled_01deg_buffered.nc"

if os.path.exists(FINAL_PATH):
    print(f"✓ Final file already exists: {FINAL_PATH}")
else:
    batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, "batch_*.nc")))
    print(f"Found {len(batch_files)} batch files to combine")

    with xr.open_dataset(batch_files[0]) as ds0:
        lat_vals = ds0["latitude"].values
        lon_vals = ds0["longitude"].values
        tvar_names = list(ds0.data_vars)

    with xr.open_dataset(STATIC_PATH) as ds_static:
        static_names = list(ds_static.data_vars)
        static_data = {v: ds_static[v].values.astype("float32") for v in static_names}

    n_lat, n_lon = len(lat_vals), len(lon_vals)
    print(f"Grid: {n_lat} lat x {n_lon} lon")
    print(f"Time-varying variables: {tvar_names}")
    print(f"Static variables: {static_names}")

    print(f"\nCreating {FINAL_PATH}...")
    out = nc4.Dataset(FINAL_PATH, "w", format="NETCDF4")
    out.createDimension("time", None)  # unlimited — can be appended to
    out.createDimension("latitude", n_lat)
    out.createDimension("longitude", n_lon)

    out.createVariable("latitude", "f4", ("latitude",))[:] = lat_vals
    out.createVariable("longitude", "f4", ("longitude",))[:] = lon_vals
    time_var = out.createVariable("time", "f8", ("time",))

    data_vars = {
        name: out.createVariable(name, "f4", ("time", "latitude", "longitude"),
                                  zlib=True, complevel=4)
        for name in tvar_names
    }
    for name in static_names:
        static_var = out.createVariable(name, "f4", ("latitude", "longitude"),
                                         zlib=True, complevel=4)
        static_var[:, :] = static_data[name]

    time_offset = 0
    for i, batch_path in enumerate(batch_files):
        with xr.open_dataset(batch_path) as ds_batch:
            batch_time = ds_batch["time"].values.astype("datetime64[s]").astype("int64")
            n_batch_t = len(batch_time)

            time_var[time_offset:time_offset + n_batch_t] = batch_time
            for name in tvar_names:
                data_vars[name][time_offset:time_offset + n_batch_t, :, :] = \
                    ds_batch[name].values.astype("float32")

        time_offset += n_batch_t
        if (i + 1) % 10 == 0 or (i + 1) == len(batch_files):
            print(f"  Appended {i + 1}/{len(batch_files)} batches ({time_offset:,} timesteps so far)")

    time_var.units = "seconds since 1970-01-01 00:00:00"
    out.close()

    size_gb = os.path.getsize(FINAL_PATH) / 1e9
    print(f"\n✓ Saved: {FINAL_PATH} ({size_gb:.2f} GB), {time_offset:,} total timesteps")

print("\nVerifying final file...")
ds_final = xr.open_dataset(FINAL_PATH)
print(f"Variables: {list(ds_final.data_vars)}")
print(f"Timesteps: {len(ds_final.time):,}")
print(f"Grid: {len(ds_final.latitude)} lat x {len(ds_final.longitude)} lon")

# ── Define the variables the downstream pickle-save cell expects ──
import pandas as pd

ds_era5 = ds_final
era5_lat = ds_era5["latitude"].values
era5_lon = ds_era5["longitude"].values
era5_time = pd.to_datetime(ds_era5["time"].values)
era5_vars = list(ds_era5.data_vars)
n_era5_cells = len(era5_lat) * len(era5_lon)

print(f"\n✓ ERA5 grid: {len(era5_lat)} lat x {len(era5_lon)} lon = {n_era5_cells:,} cells")
print(f"✓ ERA5 time steps: {len(era5_time):,}")
print(f"✓ ERA5 variables: {era5_vars}")

Found 44 batch files to combine
Grid: 141 lat x 131 lon
Time-varying variables: ['u10', 'v10', 'd2m', 't2m', 'msl', 'sst', 'sp', 'skt', 'swvl1', 'z', 'cape', 'tcwv', 'blh', 'cp', 'swvl2', 'swvl3']
Static variables: ['slt']

Creating era5_uk_upsampled_01deg_buffered.nc...
  Appended 10/44 batches (5,000 timesteps so far)
  Appended 20/44 batches (10,000 timesteps so far)
  Appended 30/44 batches (15,000 timesteps so far)
  Appended 40/44 batches (20,000 timesteps so far)
  Appended 44/44 batches (21,888 timesteps so far)

✓ Saved: era5_uk_upsampled_01deg_buffered.nc (12.20 GB), 21,888 total timesteps

Verifying final file...
Variables: ['u10', 'v10', 'd2m', 't2m', 'msl', 'sst', 'sp', 'skt', 'swvl1', 'z', 'cape', 'tcwv', 'blh', 'cp', 'swvl2', 'swvl3', 'slt']
Timesteps: 21,888
Grid: 141 lat x 131 lon

✓ ERA5 grid: 141 lat x 131 lon = 18,471 cells
✓ ERA5 time steps: 21,888
✓ ERA5 variables: ['u10', 'v10', 'd2m', 't2m', 'msl', 'sst', 'sp', 'skt', 'swvl1', 'z', 'cape', 'tcwv', 'blh', 'cp', '

In [1]:
# FIX: ADD MISSING lsm INTO THE ALREADY-COMBINED FINAL FILE
# ============================================================================
# static_vars.nc was silently reused from an earlier crashed run (before the
# lsm fix was applied), so it only ever contained 'slt' — 'lsm' never made
# it into the final combined file despite the fix being correct in the
# interpolation script. This regenerates the static file properly and adds
# lsm directly into the existing 12.2GB output, avoiding a full rebuild.
# ============================================================================
import os
import numpy as np
import netCDF4 as nc4
import xarray as xr

ERA5_PATH = "era5_uk_2024_2026_hourly_pad1.5_full.nc"
FINAL_PATH = "era5_uk_upsampled_01deg_buffered.nc"
UK_BOUNDS_FINAL = {"lat_min": 48.5, "lat_max": 62.5, "lon_min": -9.75, "lon_max": 3.25}

print("Loading source ERA5 file to extract lsm...")
ds_full = xr.open_dataset(ERA5_PATH)
time_dim = 'valid_time' if 'valid_time' in ds_full.dims else 'time'
lat_desc = ds_full["latitude"].values[0] > ds_full["latitude"].values[-1]

ds = ds_full.sel(
    latitude=slice(UK_BOUNDS_FINAL["lat_max"], UK_BOUNDS_FINAL["lat_min"]) if lat_desc
    else slice(UK_BOUNDS_FINAL["lat_min"], UK_BOUNDS_FINAL["lat_max"]),
    longitude=slice(UK_BOUNDS_FINAL["lon_min"], UK_BOUNDS_FINAL["lon_max"]),
)

new_lat = np.arange(UK_BOUNDS_FINAL["lat_min"], UK_BOUNDS_FINAL["lat_max"] + 0.05, 0.1)
new_lon = np.arange(UK_BOUNDS_FINAL["lon_min"], UK_BOUNDS_FINAL["lon_max"] + 0.05, 0.1)

lsm_source = ds["lsm"]
if time_dim in lsm_source.dims:
    lsm_source = lsm_source.isel({time_dim: 0})  # constant across time, take one snapshot

print("Interpolating lsm onto the 0.1° grid...")
lsm_upsampled = lsm_source.interp(latitude=new_lat, longitude=new_lon, method="linear",
                                   kwargs={"bounds_error": False}).values.astype("float32")
print(f"  Shape: {lsm_upsampled.shape}")

print(f"\nAdding lsm into {FINAL_PATH}...")
out = nc4.Dataset(FINAL_PATH, "a")  # append mode — modifies the existing file

if "lsm" in out.variables:
    print("  'lsm' already exists in the file — overwriting with the correct values")
    out.variables["lsm"][:, :] = lsm_upsampled
else:
    lsm_var = out.createVariable("lsm", "f4", ("latitude", "longitude"), zlib=True, complevel=4)
    lsm_var[:, :] = lsm_upsampled
    print("  ✓ Created and wrote 'lsm'")

out.close()

# ── Verify ──
print("\nVerifying...")
ds_check = xr.open_dataset(FINAL_PATH)
print(f"Variables ({len(ds_check.data_vars)}): {list(ds_check.data_vars)}")
assert "lsm" in ds_check.data_vars, "lsm still missing — something went wrong"
print("✓ lsm confirmed present")

Loading source ERA5 file to extract lsm...
Interpolating lsm onto the 0.1° grid...
  Shape: (141, 131)

Adding lsm into era5_uk_upsampled_01deg_buffered.nc...
  ✓ Created and wrote 'lsm'

Verifying...
Variables (18): ['u10', 'v10', 'd2m', 't2m', 'msl', 'sst', 'sp', 'skt', 'swvl1', 'z', 'cape', 'tcwv', 'blh', 'cp', 'swvl2', 'swvl3', 'slt', 'lsm']
✓ lsm confirmed present


In [2]:
# CELL 7: TIME FORMATTING FOR MERGING WITH IMERG DATASET, THEN PICKLE SAVE
# ============================================================================
# Explicitly reformats era5_time into an unambiguous string-based format
# before anything gets merged with IMERG downstream. This is deliberate,
# not just tidiness — this project already hit a real, hard-to-diagnose bug
# earlier where pandas 2.x didn't guarantee nanosecond datetime64
# resolution, silently breaking hour-difference calculations across several
# scripts. Relying on raw datetime64 comparison during the eventual
# ERA5<->IMERG merge risks hitting that same class of bug again if the two
# datasets' time dtypes don't match exactly. A canonical string format
# sidesteps that entirely — string equality doesn't care about underlying
# datetime64 resolution.
# ============================================================================
import pandas as pd
import numpy as np
import pickle
import os

# ── Normalize era5_time to a clean, explicit datetime index ──
era5_time = pd.to_datetime(era5_time)
print(f"era5_time dtype: {era5_time.dtype}")
print(f"Range: {era5_time.min()} to {era5_time.max()}")
print(f"Count: {len(era5_time):,}")

# Sanity check: confirm genuinely hourly and gap-free, since this exact
# assumption silently broke earlier in the project when timestamp handling
# had bugs elsewhere. Cheap to verify again here rather than assume.
diffs = era5_time.to_series().diff().dropna()
non_hourly = diffs[diffs != pd.Timedelta(hours=1)]
if len(non_hourly) > 0:
    print(f"⚠ {len(non_hourly)} non-hourly gaps found in era5_time — "
          f"investigate before merging with IMERG.")
else:
    print("✓ era5_time is fully hourly, gap-free.")

# Canonical string format for merge keys — matches the "%Y-%m-%d %H:%M:%S"
# format used throughout the rest of this project's time handling, so the
# eventual ERA5<->IMERG merge can join on exact string equality rather than
# datetime64 comparison.
era5_time_str = era5_time.strftime("%Y-%m-%d %H:%M:%S")
print(f"\nExample formatted timestamps: {list(era5_time_str[:3])} ... {list(era5_time_str[-3:])}")

# Update the xarray coordinate to the cleaned, verified time index, so
# ds_era5 and era5_time can't silently drift apart from each other
ds_era5 = ds_era5.assign_coords(time=era5_time)

# ── Package and pickle-save ──
era5_data = {
    "era5_lat": era5_lat,
    "era5_lon": era5_lon,
    "era5_time": era5_time,
    "era5_time_str": era5_time_str,  # string-formatted version for merge keys
    "era5_vars": era5_vars,
    "n_era5_cells": n_era5_cells,
    "ds_era5": ds_era5,
}

new_pickle_path = "era5_data_pad1.5_v18.pkl"
with open(new_pickle_path, "wb") as f:
    pickle.dump(era5_data, f)

print(f"\n✓ ERA5 data saved to {new_pickle_path}")
print(f"  Grid: {len(era5_lat)} lat × {len(era5_lon)} lon = {n_era5_cells:,} cells")
print(f"  Time steps: {len(era5_time):,}")

# Validate before promoting to the canonical name — small edge-of-buffer
# NaN percentage is expected and tolerated (see earlier note: no padding
# margin exists beyond the buffered domain itself)
NAN_TOLERANCE_PCT = 1.0

sample_var = era5_vars[0]
nan_count = int(np.isnan(ds_era5[sample_var].isel({"time": 0}).values).sum())
nan_pct = 100 * nan_count / n_era5_cells

print(f"\nValidation — NaN count in '{sample_var}' at first timestep: "
      f"{nan_count:,} / {n_era5_cells:,} ({nan_pct:.3f}%)")

if nan_pct <= NAN_TOLERANCE_PCT:
    if os.path.exists("era5_data.pkl"):
        os.rename("era5_data.pkl", "era5_data_OLD_backup.pkl")
        print("✓ Backed up old era5_data.pkl -> era5_data_OLD_backup.pkl")
    os.rename(new_pickle_path, "era5_data.pkl")
    print(f"✓ Promoted new pickle to era5_data.pkl "
          f"({nan_pct:.3f}% NaN, within {NAN_TOLERANCE_PCT}% tolerance)")
    if nan_count > 0:
        print("  Note: confirm remaining NaNs are concentrated at the buffered domain's "
              "outer edge (expected) rather than scattered throughout (a real problem) "
              "before proceeding to the IMERG merge.")
else:
    print(f"⚠ NOT promoting — {nan_pct:.3f}% NaN exceeds the {NAN_TOLERANCE_PCT}% tolerance. "
          f"Investigate before overwriting the working pickle.")

NameError: name 'era5_time' is not defined